In [ ]:
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu130
%pip install scikit-learn torch.utils pandas matplotlib

# PyTorch Classification Mock Model w/ Credit Data
## (INTENDED) Goal

Build a classification model using `pytorch` upon the `Credit Risk Kaggle Dataset`. Through various steps, the data will be laundered, then put into `CreditRisk` classes. These will be put into `DataLoader`s and then interpreted by the classification model. The end goal is to determine whether or not an individual, based on given traits poses no credit risk (0) or is likely to default (1). 

## Tasks: 
1. **Setup & Data Loading:** 
    - Import `torch`, `pandas`, `sklearn`, and other dependencies.
    - Load in the dataset from `.csv`.
    - Check for cuda.

2. **Explore Data and Clean:**
    - Use `.info()`, `.describe()`, `.isnull()`, and other to find missing data/read data. 
    - Use `IterativeImputer` and/or `mean/median` to fill missing values within the dataset. 
    - Remove outliers that skew the data. i.e.: People with extensive employment lengths (60 years) or extreme age values (100 years)

3. **Feature Engineer and Preprocess Data**
    - Convert target from y/n to 1/0
    - Transform text features into numerical columns
    - Define features X and target y

4. **Data Splitting and Scaling:**
    - Split the data using `train_test_split` into training and testing sets (80/20) w/ random state (42)
    - Apply Standard Scaler to both feature sets to normalize the inputs

5. **PyTorch It!**
    - Convert numpy arrays into torch float tensors
    - Build `CreditDataset`
    - Use DataLoaders to create training and testing iterations with batch sizing

6. **Model Creation:**
    - Define your class with `nn.Linear`, `nn.ReLU`, `nn.Dropout`, and `nn.BatchNorm1d`
    - Initialize loss function. 
    - Set up Adam Optimizer with learning rate and weight decay

7. **Training Loop/Validation:**
    - Implement a training loop with the forward pass, loss calc, backprop, and weight updating
    - Check classification accuracy + loss during training
    - Check validation loss -> Check for overfitting

8. **Evaluation:**
    - Run the model in `eval()` mode. 
    - Generate performance report with Confusion matrix, Precision, recall, and f1-score

9. **Visualization + Analysis:**
    - Plot training vs validation loss over epochs
    - Visualize model mistakes through confusion heatmap

10. **Model Saving & Data Storage:**
    - Save using `torch.save()` to store the trained model weights
    - Save the `StandardScaler` using joblib to reuse it later on

## Setup & Data Loading
**Tasks:**
1. Import libraries
2. Check for Cuda availability
3. Load Credit Risk Dataset from `.csv` file within `../data/raw/`

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [ ]:
# Check if Cuda is Available
torch.cuda.is_available()

# View Primary GPU
torch.cuda.get_device_name(0)

# Set default torch device to GPU (cuda)
torch.set_default_device('cuda')

In [ ]:
# Reading the csv file
credit_risk_df = pd.read_csv("../data/raw/credit_risk_dataset.csv")

# Printing Top 5 Rows
print(credit_risk_df.head())
print("-------------------------------------------------------------------------------------------------------")

## Explore Data 
**Tasks:**
1. Use `.info()`, `.describe()`, and `.isnull()` to find missing values within the dataframe

In [ ]:
print(credit_risk_df.info())
print(credit_risk_df.describe())

In [21]:
# Shape and Size of the Data
print(credit_risk_df.size)
print(credit_risk_df.shape)

# Check for null values
print(credit_risk_df.isnull().sum())

390972
(32581, 12)
person_age                       0
person_income                    0
person_home_ownership            0
person_emp_length              895
loan_intent                      0
loan_grade                       0
loan_amnt                        0
loan_int_rate                 3116
loan_status                      0
loan_percent_income              0
cb_person_default_on_file        0
cb_person_cred_hist_length       0
dtype: int64


## Encode Features & Clean Data
**Tasks:**
1. Encode target variable with `LabelEncoder()`
2. Encode categorical features with `pd.get_dummies()`
3. Use `IsolationForest` to find and separate clear outliers

In [22]:
# First we must encode the target var
# We will use LabelEncoder from sklearn
lb_enc = LabelEncoder()

credit_risk_df['cb_person_default_on_file'] = lb_enc.fit_transform(credit_risk_df['cb_person_default_on_file'])
print(credit_risk_df['cb_person_default_on_file'])

0        1
1        0
2        0
3        0
4        1
        ..
32576    0
32577    0
32578    0
32579    0
32580    0
Name: cb_person_default_on_file, Length: 32581, dtype: int64


In [ ]:
# Encode Categorical Features With get_dummies()


In [ ]:
# Thinking of iterative imputer

#Our imports
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge

credit_risk_encoding